In [ ]:
import json

import polars as pl
from tqdm import tqdm

from vlm_diagram_eval import compat
from vlm_diagram_eval.evaluators.metrics import *
from vlm_diagram_eval.parsing import graph as parser_service

In [ ]:
model_name = "o4-mini"
prompt_version = "v3"
reasoning = "medium"

In [ ]:
df_900 = pl.read_parquet(
    f"inference_image2mermaid/data_900_prompt_{prompt_version}_gpt-{model_name}_reasoning_{reasoning}.parquet"
)

In [ ]:
evaluator = DirectedSpectralSimilarity()

In [ ]:
evaluator.name()

In [ ]:
def calculate_scores_and_stats(input_df: pl.DataFrame, evaluator, results_path: str, stats_path: str):
    """Calculates evaluation scores for each row, saves detailed results, and computes summary statistics."""

    is_directed_error = evaluator.name() == "DirectedErrorEvaluator"

    f1_scores = []
    jaccard_scores = []
    all_metrics = []

    scores = []

    print(f"Calculating {evaluator.name()} for each row...")
    for row in tqdm(input_df.iter_rows(named=True), total=len(input_df)):
        gt_code = row.get("code")
        gen_code = row.get("generated_code")

        score = None
        try:
            if not gen_code:
                score = 0.0 if not is_directed_error else {}
            else:
                try:
                    parser_service.get_graph_from_json(gen_code)
                except Exception:
                    score = 0.0 if not is_directed_error else {}
                else:
                    score = evaluator.evaluate(gt_code, gen_code) if gt_code else (0.0 if not is_directed_error else {})
        except Exception as e:
            print(f"Unexpected error for {row.get('filename', 'N/A')}: {e}")
            score = 0.0 if not is_directed_error else {}

        if is_directed_error:
            f1_scores.append(score.get("Score_F1", 0.0) if isinstance(score, dict) else 0.0)
            jaccard_scores.append(score.get("Score_Jaccard", 0.0) if isinstance(score, dict) else 0.0)
            all_metrics.append(json.dumps(score) if isinstance(score, dict) else "{}")
        else:
            scores.append(score)

    # Add the new scores as columns to the original DataFrame
    if is_directed_error:
        results_df = input_df.with_columns(
            [
                pl.Series("f1_score", f1_scores, dtype=pl.Float64),
                pl.Series("jaccard_score", jaccard_scores, dtype=pl.Float64),
                pl.Series("all_metrics", all_metrics, dtype=pl.String),
            ]
        )
    else:
        results_df = input_df.with_columns(pl.Series(name=evaluator.name(), values=scores, dtype=pl.Float64))

    print(f"\nSaving detailed results with spectral scores to {results_path}...")
    results_df.write_parquet(results_path)

    # --- Compute Statistics ---
    print(f"Computing summary statistics for {evaluator.name()}...")
    if is_directed_error:
        stats_df = results_df.group_by(["diagram_type", "difficulty"]).agg(
            [
                pl.col("f1_score").mean().alias("avg_f1_score"),
                pl.col("jaccard_score").mean().alias("avg_jaccard_score"),
                pl.col("f1_score").std().alias("std_f1_score"),
                pl.col("jaccard_score").std().alias("std_jaccard_score"),
                pl.col("f1_score").drop_nulls().len().alias("n_valid_scores"),
                pl.len().alias("total_count"),
            ]
        )
    else:
        stats_df = results_df.group_by(["diagram_type", "difficulty"]).agg(
            [
                pl.col(evaluator.name()).mean().alias(f"avg_{evaluator.name()}"),
                pl.col(evaluator.name()).std().alias(f"std_{evaluator.name()}"),
                pl.col(evaluator.name()).min().alias(f"min_{evaluator.name()}"),
                pl.col(evaluator.name()).max().alias(f"max_{evaluator.name()}"),
                pl.col(evaluator.name()).drop_nulls().len().alias("n_valid_scores"),
                pl.len().alias("total_count"),
            ]
        )

    print(f"Saving statistics to {stats_path}...")
    stats_df.write_parquet(stats_path)
    return results_df, stats_df

In [ ]:
# Define output paths
#
RESULTS_FILE = f"results/results_{evaluator.name()}_{prompt_version}_{model_name}_reasoning_{reasoning}.parquet"

STATS_FILE = f"results/stats_{evaluator.name()}_{prompt_version}_{model_name}_reasoning_{reasoning}.parquet"

# Run the processing
results_df, stats_df = calculate_scores_and_stats(df_900, evaluator, RESULTS_FILE, STATS_FILE)

In [ ]:
stats_df